# Lasmoid: 100M Parameter Model Training Pipeline on Kaggle

This notebook demonstrates how to clone, configure, and train the **most powerful 100M parameter version** of the **Lasmoid Hybrid Concept Transformer** model on a Kaggle GPU instance (e.g. T4 GPU or P100).

### Core Architectural Features in this 100M Config:
- **Tied Embedding/Projection weights**: Vocabulary matrices are shared to fit within the 100M threshold (vocabulary takes 49.6M parameters).
- **12 Interleaved Layers**: Deeper stack of CSA (Compressed Sparse Attention) and parallel Mamba-3 SSM layers.
- **Grey-Box Mixture of Experts (MoE)**: 6 routed experts and 1 shared expert (simulates ~300M parameter capacity with a fast active footprint).
- **Increased Context & Reasoning**: `max_seq_len = 512` and `reasoning_steps = 2` to accommodate deep reasoning budgets using `<think>...</think>` tokens.
- **Muon & AdamW Optimizer**: Muon for 2D weights (Newton-Schulz orthogonalization) and AdamW for scales, biases, and embeddings.

## Step 1: Environment Setup & Codebase Retrieval

In [ ]:
# Install dependencies (Kaggle environments already have PyTorch installed)
!pip install -q transformers datasets tqdm

# Clone the official Lasmoid codebase
!git clone https://github.com/Theory903/Lasmoid.git
%cd Lasmoid

## Step 2: Define the 100M Parameter Configuration

We update the configuration with the optimized parameters: `dim = 384`, `n_layers = 12`, `max_seq_len = 512`, and 7 total experts (6 routed + 1 shared) to reach **97.9M parameters**.

In [ ]:
import json

config_100m = {
  "vocab_size": 129286,
  "max_seq_len": 512,
  "max_batch_size": 4,
  "dtype": "bf16",
  "norm_eps": 1e-06,
  "rope_theta": 10000.0,
  "rope_factor": 1.0,
  "beta_fast": 32,
  "beta_slow": 1,
  "original_seq_len": 0,
  "swiglu_limit": 10.0,
  "num_residual_streams": 4,
  "hc_sinkhorn_iters": 8,
  "hc_eps": 1e-06,
  "hcm_ema_alpha": 0.99,
  "hcm_commit_loss_coeff": 0.25,
  "entropy_threshold": 0.5,
  "router_z_loss_coeff": 0.001,
  "ema_bias_lr": 0.01,
  "expert_capacity_factor": 1.25,
  "moe_load_balance_coeff": 0.01,
  "predictive_coding_coeff": 0.01,
  "reasoning_steps": 2,
  "think_token_id": 128821,
  "answer_token_id": 129285,
  "cot_exit_confidence": 0.9,
  "moe_router_entropy_coeff": 0.001,
  "moe_capacity_loss_coeff": 0.01,
  "token_concept_loss_coeff": 0.05,
  "steering_attributes": [
    "creativity",
    "helpfulness",
    "complexity",
    "scientific_rigor"
  ],
  "post_attn_norm": True,
  "post_ffw_norm": True,
  "moe_dual_ffn": True,
  "ssm_d_skip": True,
  "dim": 384,
  "n_layers": 12,
  "n_heads": 6,
  "q_lora_rank": 96,
  "head_dim": 48,
  "rope_head_dim": 16,
  "o_groups": 2,
  "o_lora_rank": 96,
  "n_routed_experts": 6,
  "n_shared_experts": 1,
  "n_activated_experts": 2,
  "moe_latent_dim": 192,
  "num_concepts": 64,
  "num_abstract_concepts": 8,
  "num_global_concepts": 2,
  "codebook_size": 256,
  "lightning_topk_blocks": 2,
  "ssm_heads": 6,
  "ssm_state_dim": 16,
  "ssm_kernel_size": 4,
  "ssm_chunk_size": 64,
  "ssm_dt_min": 0.001,
  "ssm_dt_max": 0.1,
  "ssm_dt_init_floor": 0.0001,
  "ssm_n_groups": 1
}

# Overwrite config.json with the 100M parameter version
with open("config.json", "w") as f:
    json.dump(config_100m, f, indent=2)

print("config.json updated successfully for 100M parameter version.")

## Step 3: Verify the Parameter Count

Let's initialize the model structure on the meta device to verify the parameter count and tensor layout before allocating memory.

In [ ]:
import torch
from inference.model import Lasmoid, ModelArgs

with open("config.json") as f:
    config_dict = json.load(f)

args = ModelArgs(**config_dict)

# Count on meta device (no memory allocated)
with torch.device("meta"):
    model = Lasmoid(args)
total_params = sum(p.numel() for p in model.parameters())
print(f"Model Dimension: {args.dim}")
print(f"Number of Layers: {args.n_layers}")
print(f"Model Parameters: {total_params:,}")

## Step 4: Run Training

We will train the 100M model on the GPU. Since the model has larger activations (due to sequence length 512 and dim 384), we use a micro-batch size of `2` paired with gradient accumulation (`--grad_accum 4`) to yield an effective batch size of `8` while preventing out-of-memory (OOM) errors on 16GB GPUs.

In [ ]:
# Run training loop on Kaggle GPU
!python train/train.py \
  --max_iters 1000 \
  --batch_size 2 \
  --grad_accum 4 \
  --learning_rate 4e-4 \
  --device cuda

## Step 5: Test Text Generation

Once training completes or gets interrupted (with checkpoint saved), we test text generation using the generated checkpoint.

In [ ]:
with open("test_prompts.txt", "w") as f:
    f.write("To be, or not to be\n")

# Run generation in batch mode
!python inference/generate.py \
  --ckpt-path checkpoints/current/lasmoid_final.pt \
  --config config.json \
  --input-file test_prompts.txt \
  --max-new-tokens 50 \
  --device cuda